# Critical candidate and bounded bubble continuation

These calculations use Gross and Sadowski (2001) pure propane and methane/ethane
packets from the source archive. Temperature is K, pressure Pa, density mol/m³,
and feed amounts mol. They verify native equations, not physical measurements.
A critical-equation candidate has no global stability certificate. The binary
path below covers only the declared bubble branch from 180 to 200 K, not a
complete envelope through a critical point. Physical uncertainty is unquantified.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import Parameters, Pure, Mixture, equilibrium as eq

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ARCHITECTURE.yaml").is_file())
packets = ROOT / "data" / "packets"
propane = Pure(Parameters.from_packets(packets, [("gross-2001-propane/3", ["propane"])]))
critical = eq.critical_point(propane, T_guess=375, P_guess=1e6, rho_guess=5000)
assert critical.temperature is not None
assert abs(critical.temperature / 375.14002748147936 - 1) < 1e-8
assert critical.critical is not None
assert abs(critical.critical.eigenvalues[0]) <= critical.critical.zero_band
display(pd.DataFrame({"T / K": [critical.temperature], "P / Pa": [critical.pressure],
                      "Maximum classified residual": [max(map(abs, critical.classified_residuals))]}))


,T / K,P / Pa,Maximum classified residual
0,375.140027,4.607730e+06,8.881784e-15


In [2]:
mixture = Mixture(Parameters.from_packets(
    packets, [("gross-2001-methane-ethane/3", ["methane", "ethane"])]))
feed = eq.Amounts({"methane": 0.3, "ethane": 0.7})
start = eq.bubble_point(liquid=eq.Phase("L", kind="liquid"), T=180, feed=feed)
target = eq.bubble_point(liquid=eq.Phase("L", kind="liquid"), T=200, feed=feed)
options = eq.Continuation(initial_step=0.1, minimum_step=0.001,
                          max_steps=512, max_retries_per_step=8)
path = eq.continue_equilibrium(mixture, start, target, options)
assert path.success and path.reached_path == 1.0, path.stop
accepted = pd.DataFrame({"Path": path.path_values,
    "T / K": [s.temperature for s in path.states],
    "P / Pa": [s.pressure for s in path.states],
    "Maximum classified residual": [max(map(abs, s.classified_residuals)) for s in path.states]})
assert abs(accepted.iloc[-1]["T / K"] - 200) < 1e-8
assert accepted["Maximum classified residual"].max() <= 1e-10
display(accepted)


,Path,T / K,P / Pa,Maximum classified residual
0,0.0000,180.00,1.016243e+06,1.065814e-14
1,0.0125,180.25,1.023455e+06,2.664535e-15
2,0.0250,180.50,1.030698e+06,3.108624e-15
3,0.0375,180.75,1.037973e+06,8.437695e-15
4,0.0500,181.00,1.045279e+06,5.773160e-15
...,...,...,...,...
77,0.9625,199.25,1.666419e+06,2.664535e-15
78,0.9750,199.50,1.676160e+06,9.769963e-15
79,0.9875,199.75,1.685934e+06,1.680999e-15
80,1.0000,200.00,1.695743e+06,5.329071e-15


Prediction, correction and retries run natively. The table retains every
accepted temperature, pressure and classified residual; `reached_path` must be
one before the final state can be called the target. No branch completeness or
mixture critical-crossing claim follows from this bounded path.
